In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql

SELECT * FROM catalog.bronze.diagnosis_raw;

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
bronze_table = 'catalog.bronze.diagnosis_raw'
silver_table = 'catalog.silver.dim_diagnosis'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_diagnosis/checkpoint/"

# Read from Bronze as a stream
df_diagnosis_bronze = spark.readStream.table(bronze_table)

df_diagnosis_clean = (
    df_diagnosis_bronze
        .drop(
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumn("load_timestamp", current_timestamp())
)

from delta.tables import DeltaTable

# When diagnosis codes match, data gets merged. If not, data gets updated
def merge_dim_diagnosis(batch_df, batch_id):
    batch_df = batch_df.dropDuplicates(["diagnosis_code"])

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))   
        return
    
    # Load Delta table by name and upsert into Silver
    dim_diagnosis = DeltaTable.forName(spark, silver_table)

    (dim_diagnosis.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.diagnosis_code = s.diagnosis_code"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(
    df_diagnosis_clean.writeStream
        .foreachBatch(merge_dim_diagnosis)  # for every batch, merge_dim_diagnosis is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)


In [0]:
%sql
SELECT * FROM catalog.silver.dim_diagnosis;

In [0]:
# Data quality check
from pyspark.sql.functions import col, count

print("DIM DIAGNOSIS")
df = spark.read.table("catalog.silver.dim_diagnosis")
total = df.count()
print(f"Total rows: {total} (expected: 28)")

# Uniqueness
dup_id = total - df.select("diagnosis_code").distinct().count()
print(f"Duplicate diagnosis_codes: {'OK' if dup_id == 0 else dup_id}")

# Nulls
for c in ["diagnosis_code", "diagnosis_desc", "diagnosis_category", "readmission_risk_level"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

# Valid risk levels
print("Risk level distribution:")
df.groupBy("readmission_risk_level").count().show()